# PUNCH Sandbox Demo: From Images to J-maps and CME Kinematics

This notebook walks through one complete analysis path using PUNCH Level 3 FITS data:

1. retrieve or point to a sequence of PUNCH observations,
2. load one image and convert it into polar coordinates,
3. repeat that transform for a time sequence,
4. build a stack-plot/J-map to track outward-moving structure, and
5. estimate kinematics and intensity evolution from user-selected points.

The notebook is intentionally interactive. Several cells produce plots that are meant to be discussed live during a session.

### What background is helpful?

You do not need to be a PUNCH software expert to follow this notebook, but it helps to know:
- what FITS image products are,
- the idea of mapping coronal structure into polar coordinates, and
- how a stack-plot/J-map represents brightness as a function of time and elongation.

### Before you run the notebook

- This notebook assumes `punchbowl`, `sunpy`, `astropy`, and the interactive plotting backend are available.
- The examples below use a specific date range as a demo dataset. You can swap in a different observing interval later.
- Interactive point-picking works best when `%matplotlib widget` is active.


## Environment Setup and Imports

The next few cells handle optional package installation, optional Google Drive mounting for Colab users, and the imports used throughout the analysis.

If you are running locally and already have the required packages installed, you can usually skip the installation cell.


In [ ]:
# Uncomment and run this cell only when you are in Google Collabe but not locally if
# these packages are installed beforrehand.

!pip install ipympl
# !pip install punchbowl
!pip install git+https://github.com/punch-mission/punchbowl.git
!pip install ipywidgets==7.7.1  # To get tqdm working with Google Colab

  Cloning https://github.com/punch-mission/punchbowl.git to /tmp/pip-req-build-b32ty3li
  Running command git clone --filter=blob:none --quiet https://github.com/punch-mission/punchbowl.git /tmp/pip-req-build-b32ty3li
  Resolved https://github.com/punch-mission/punchbowl.git to commit df2ac10b8c29aebf5e4b5db19b40a52d371fe7da
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
Using cached widgetsnbextension-4.0.15-py3-none-any.whl (2.2 MB)
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipy

  Using cached ipywidgets-7.7.1-py2.py3-none-any.whl.metadata (1.9 kB)
  Using cached widgetsnbextension-3.6.10-py2.py3-none-any.whl.metadata (1.3 kB)
Using cached ipywidgets-7.7.1-py2.py3-none-any.whl (123 kB)
Using cached widgetsnbextension-3.6.10-py2.py3-none-any.whl (1.6 MB)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
^C


In [ ]:
# Uncomment and run this cell to mount your own google drive for persistent data
# You will be prompted to give access to your Google Account and Google Drive.

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Choose an interactive matplotlib backend for notebook use.
# `%matplotlib widget` is recommended here because later cells rely on mouse clicks.
# If you are running in a different environment, `%matplotlib inline` may be safer,
# but the interactive point-picking cells will then need to be adapted.
# %matplotlib tk
# %matplotlib inline

%matplotlib widget

# Some imports to support Colab
from google.colab import output
output.enable_custom_widget_manager()

# Standard scientific Python imports.
import os
import glob
import numpy as np
import astropy.units as u
import matplotlib.dates as mdates

from tqdm.notebook import tqdm
from matplotlib import pyplot as plt
from matplotlib.colors import PowerNorm
from astropy.time import Time
from astropy.constants import au
from ndcube import NDCube
from sunpy.net import Fido
from sunpy.net import attrs as a
from scipy.ndimage import percentile_filter, median_filter
from scipy.interpolate import CubicSpline
from scipy.signal import savgol_filter

# Import punchbowl first so the PUNCH-specific Fido search attributes are registered.
import punchbowl  # Needed so Fido recognizes the PUNCH search attributes used below.
from punchbowl.data import punch_io, visualize
from punchbowl.data.punch_io import load_many_cubes, load_ndcube_from_fits
from punchbowl.data.visualize import cmap_punch
from punchbowl.level3.velocity import preprocess_image

# Widgets are used later for the point-selection workflow.
from IPython.display import display
import ipywidgets as widgets

import warnings
warnings.filterwarnings('ignore')


## Step 1: Select and Access PUNCH Data

This section shows one way to query the PUNCH archive with SunPy `Fido`. The search is restricted by observing time, product code, instrument, processing level, version, and file type so that the returned files are scientifically consistent for the demo.


### Query the archive with `Fido`

This cell performs a metadata search only. It does not download the files yet. Review the returned table first so you can confirm that the requested interval and product selection match the event you want to discuss.


In [ ]:
# Search the archive for a specific PUNCH observing window.

result = Fido.search(a.Time('2025/09/22 03:00:00', '2025/09/23 09:59:59'),
                     a.punch.ProductCode.ca,  # `ca` = clear low-noise. 'pa' for polarized low-noise.
                     a.Instrument.m,          # `m` = mosaic instrument product.
                     a.Level.three,
                     a.punch.DataVersion.newest,  # Use the newest available processing version.
                     a.punch.FileType.fits)       # FITS is convenient for analysis; JP2 is another option.

# Display the search result table so you can inspect what will be downloaded.
result


Start Time,End Time,Level,ProductCode,Instrument,DataVersion,Source,Provider,FileType
Time,Time,str1,str2,str1,str2,str5,str4,str4
2025-09-22 03:28:00.000,2025-09-22 03:28:00.999,3,CA,M,0k,PUNCH,SwRI,fits
2025-09-22 04:00:00.000,2025-09-22 04:00:00.999,3,CA,M,0k,PUNCH,SwRI,fits
2025-09-22 04:32:00.000,2025-09-22 04:32:00.999,3,CA,M,0k,PUNCH,SwRI,fits
2025-09-22 05:04:00.000,2025-09-22 05:04:00.999,3,CA,M,0k,PUNCH,SwRI,fits
2025-09-22 05:36:00.000,2025-09-22 05:36:00.999,3,CA,M,0k,PUNCH,SwRI,fits
2025-09-22 06:08:00.000,2025-09-22 06:08:00.999,3,CA,M,0k,PUNCH,SwRI,fits
2025-09-22 06:40:00.000,2025-09-22 06:40:00.999,3,CA,M,0k,PUNCH,SwRI,fits
2025-09-22 07:12:00.000,2025-09-22 07:12:00.999,3,CA,M,0k,PUNCH,SwRI,fits
2025-09-22 07:44:00.000,2025-09-22 07:44:00.999,3,CA,M,0k,PUNCH,SwRI,fits


In [ ]:
# Download the files returned by the search.
# If the search returned no matches, the IndexError branch keeps the notebook from failing abruptly.
try:
    files = Fido.fetch(result[0][:])
except IndexError:
    print("Oops no files were found!")
    files = None


Files Downloaded:   0%|          | 0/58 [00:00<?, ?file/s]

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

Support for third party widgets will remain active for the duration of the session. To disable support:

In [ ]:
from google.colab import output
output.disable_custom_widget_manager()

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

Support for third party widgets will remain active for the duration of the session. To disable support:

In [ ]:
from google.colab import output
output.disable_custom_widget_manager()

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

Support for third party widgets will remain active for the duration of the session. To disable support:

In [ ]:
from google.colab import output
output.disable_custom_widget_manager()

Support for third party widgets will remain active for the duration of the session. To disable support:

In [ ]:
from google.colab import output
output.disable_custom_widget_manager()

#### Alternative: use previously downloaded files

If you already have a local cache of PUNCH FITS files, you can skip the `Fido.fetch(...)` step and point the notebook to that directory instead.

In [ ]:
# Uncomment these lines and point to a directory that already contains the PUNCH FITS files.
# This avoids re-downloading data every time you run the notebook.

# path = '/Users/rpatel/sunpy/data/'
# files = sorted(glob.glob(path + '*20250922*.fits'))


#### Quick sanity check on the file list

The next cell displays one example filename from the downloaded list. This is simply a lightweight check that the search and fetch steps returned something sensible before we start the scientific analysis.


In [ ]:
# Show one example file path from the list so we know the search/fetch step succeeded.
files[24]

'/root/sunpy/data/PUNCH_L3_CAM_20250922161600_v0k.fits'

## Single-Image Example: Cartesian to Polar

We begin with one PUNCH image so the geometry is easy to interpret. The goal is to convert the image from its native projection into a polar representation where:
- the horizontal axis is position angle or azimuth, and
- the vertical axis is elongation or radial distance from Sun center.

This representation is convenient for tracking structures that propagate outward through time.


In [ ]:
# Start with one representative file from the sequence.
# This gives us a clean single-image example before scaling up to the full time series.
# We use the punchbowl NDCube loader so the image, metadata, and WCS stay together.
input_data = load_ndcube_from_fits(files[29])

# Convert the metadata into a standard FITS header.
# That makes it easier to inspect the geometric quantities used later in the notebook.
full_header = input_data.meta.to_fits_header(wcs=input_data.wcs,
                                             write_celestial_wcs=not False)


In [ ]:
print(np.shape(input_data))
print(input_data.data)

(4096, 4096)
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [ ]:
# Inspect the FITS header if you want to connect the analysis steps to the underlying metadata.
# This is a good place to notice quantities such as solar radius, observer distance,
# plate scale, and time stamp in the output below.
full_header


SIMPLE  = 'T       '           / Conforms to FITS Standard                      
BITPIX  =                  -32 / Number of bits per pixel                       
NAXIS   =                    2 / Number of axes                                 
NAXIS1  =                 4096 / Length of the first axis                       
NAXIS2  =                 4096 / Length of the second axis                      
EXTNAME = 'PRIMARY DATA ARRAY' / Name of this binary table extension            
LONGSTRN= 'OGIP 1.0'           / The OGIP long string convention may be used    
COMMENT ----- Documentation, Contact, and Collection Metadata ------------------
DOI     = 'https://doi.org/10.48322/nnv7-bn21' / Data reference DOI             
PROJECT = 'PUNCH   '                                                            
TITLE   = 'PUNCH Level-3 Unpolarized Mosaic'                                    
KEYVOCAB= 'Unified Astronomy Thesaurus Keywords'                                
KEYWORDS= 'Solar Corona (148

## Polar-remap workflow adapted from Raphael's December 2025 sandbox

The next few cells adapt an existing sandbox workflow into a more explicit step-by-step form for demonstration purposes.


### Geometry and Polar-Remapping Parameters

These cells define a few geometric conversion factors and then choose the sampling used for the polar remap.

The exact binning is somewhat analysis-dependent. For a demo, the key idea is that we are choosing a radial extent and an azimuthal sampling that preserve enough detail to make out outward-moving features clearly.


In [ ]:
# Convert a few commonly used geometric quantities into convenient units.
RS_ARCSEC = full_header["RSUN_ARC"] * u.arcsec   # Apparent solar radius as seen by the observer.
ARCSEC_RAD = RS_ARCSEC.to(u.rad)                 # Same quantity in radians.
AU_KM = au.to(u.km)                              # Astronomical Unit in kilometers.
ARCSEC_KM = ARCSEC_RAD * AU_KM                   # Approximate km scale corresponding to 1 arcsec at 1 AU.


In [ ]:
# ------------ Parameters related to the polar transform ------------ #

# The polar remap will extend from Sun center out to 45 degrees in elongation.
max_radius_deg = 45

# Number of radial samples implied by that range and the native image pixel scale.
polar_nr = max_radius_deg / full_header["CDELT1"]

# Choose the azimuthal sampling of the remapped image.
# This is a practical compromise for the demo, not a formally optimized setting.
num_azimuth_bins = int(1440 * 8)
az_bin = 4  # Average together every 4 azimuth bins after remapping.

# Sanity check: the averaging factor must divide the azimuthal dimension evenly.
assert num_azimuth_bins % az_bin == 0
polar_naz = int(num_azimuth_bins / az_bin)

# Optional preprocessing ideas are left commented out here for experimentation.
# input_data.data[...] = percentile_filter(input_data.data[...], 50, 10)

# Remap the image into polar coordinates.
polar_image, polar_header = preprocess_image(
    input_data,
    max_radius_deg / full_header["CDELT1"],
    num_azimuth_bins,
    az_bin,
    normalize=False,
)

# Derive convenient scale conversions for later interpretation.
arcsec_per_px = 45 * 3600 / polar_image.shape[0]
km_per_px = arcsec_per_px * ARCSEC_KM
deg_per_px = 360 / polar_image.shape[1]
print('Image dimensions', polar_image.shape)
print('Radial pixel scale in deg/px:', full_header["CDELT1"])
print('Azimuthal pixel scale in deg/px: ', deg_per_px)


Image dimensions (2000, 2880)
Radial pixel scale in deg/px: 0.0225
Azimuthal pixel scale in deg/px:  0.125


In [ ]:
print(np.shape(polar_image))
print(polar_image)

(2000, 2880)
[[0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 ...
 [1.48836628e-14 5.32431636e-15 6.03881929e-15 ... 4.69679790e-15
  7.58225962e-15 1.70660071e-14]
 [1.42682197e-14 9.15990230e-15 7.51331512e-15 ... 2.91503349e-15
  8.10499253e-15 1.34623114e-14]
 [1.18447797e-14 1.15389260e-14 9.78796554e-15 ... 4.43840957e-15
  6.83049788e-15 9.71920965e-15]]


In [ ]:
fontsize = 16

# --- Build the two-panel figure ---
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Panel 1: Catesian PUNCH image
axes[0].imshow(input_data.data, origin='lower', cmap=cmap_punch,
            aspect='equal', vmin=1e-15, vmax=1e-13)
axes[0].set_title(f"Cartesian coords image\n{polar_header['DATE-OBS']}", fontsize=fontsize)
axes[0].set_xlabel('X [Pixels]', fontsize=fontsize)
axes[0].set_ylabel('Y [Pixels]', fontsize=fontsize)
axes[0].tick_params(axis='both', which='major', labelsize=fontsize)

# Panel 2: polar-remapped image
# In this view, outward-moving features are often easier to isolate by angle.
vmin = np.percentile(polar_image, 5)
vmax = np.percentile(polar_image, 98)
extent_polar = [0, 359, 0, 45]

axes[1].imshow(polar_image, origin='lower', cmap=cmap_punch,
               norm=PowerNorm(gamma=1/2.2, vmin=vmin, vmax=vmax),
               extent=extent_polar, aspect='auto')
axes[1].set_title(f"Polar-remapped image\n{polar_header['DATE-OBS']}", fontsize=fontsize)
axes[1].set_xlabel('Azimuth [degree]', fontsize=fontsize)
axes[1].set_ylabel('Radial [degree]', fontsize=fontsize)
axes[1].tick_params(axis='both', which='major', labelsize=fontsize)

#print(polar_image)

plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

In [ ]:
np.save('polar_remapped.npy', polar_image)

In [ ]:
!ls

drive  polar_remapped.npy  sample_data


## Step 2: Extend the polar transform to the full time sequence

This is where the notebook transitions from a single-image example to a time-series analysis product.


## Image Sequence: Build a Time Series in Polar Coordinates

After validating the method on one frame, we repeat the same remapping for every file in the sequence. We then collapse a selected azimuth range into a one-dimensional radial profile for each time step.

Stacking those profiles in time produces the data structure we use to build a stack-plot/J-map.


In [ ]:
# Loop over all files, remap each image to polar coordinates, and reduce a chosen
# angular sector to a single radial profile. Those profiles will later be stacked in time.
distance_time_map = []
cubeall = []
hdrcube = []

# Define the position-angle window to summarize.
# Adjust these numbers if you want to follow a different structure.
angle_range = (np.array([250.0, 260.0]) / deg_per_px).astype(int)

for file in tqdm(files[0:len(files)]):
    cube = load_ndcube_from_fits(file)
    cube.data[...] = cube.data[...]
    polar_image, polar_header = preprocess_image(
        cube,
        max_radius_deg / full_header["CDELT1"],
        num_azimuth_bins,
        az_bin,
        normalize=False,
    )
    cubeall.append(polar_image)

    # Collapse the selected angular wedge to one radial profile per time step.
    distance_time_map.append(np.median(polar_image[:, angle_range[0]:angle_range[1]], axis=1))
    hdrcube.append(cube.meta.to_fits_header(wcs=cube.wcs, write_celestial_wcs=not False))

# Transpose into the orientation expected by imshow: radial coordinate on y, time on x.
distance_time_map = np.asarray(distance_time_map).T


  0%|          | 0/58 [00:00<?, ?it/s]

In [ ]:
# Check the size of the assembled time-distance array.
distance_time_map.shape


(2000, 58)

### Choose Any Other Angular Sector of Interest

This alternate extraction step lets you define a different position-angle window after the polar images have already been generated. That is useful during a demo because you can quickly retarget the analysis toward a visible outflow or CME without re-downloading the data.


In [ ]:
# If the polar-remapped cube is already in memory, you can quickly define a new
# position-angle window and rebuild the time-distance product without repeating the full download step.
distance_time_map2 = []
angle_range = (np.array([235.0, 245.0]) / deg_per_px).astype(int)
for i in tqdm(range(len(cubeall))):
    distance_time_map2.append(
        np.nanmedian(np.asarray(cubeall[i].data)[:, angle_range[0]:angle_range[1]], axis=1)
    )

distance_time_map2 = np.asarray(distance_time_map2).T


## Step 3: Build and interact with a stack-plot a.k.a. J-map

A stack-plot displays brightness as a function of time and elongation. Outward-moving structures often appear as sloped tracks.

What features can you identify here are likely solar outflows, background structure, or processing artifacts?


In [ ]:
# Build the stack-plot axes using observation times from the FITS headers.
# The x-axis is time and the y-axis is elongation angle in degrees.
# Left-click to add points along a visible track. Use the button to finalize the selection.

times = Time([h["DATE-OBS"] for h in hdrcube], format="isot", scale="utc").to_datetime()
t_num = mdates.date2num(times)
extent = [t_num.min(), t_num.max(), 0, 45]
input_map = distance_time_map2
vmin = 5e-15 #np.percentile(input_map, 30)
vmax = 7.5e-13 #np.percentile(input_map, 99)

fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(
    input_map,
    origin="lower",
    cmap=cmap_punch,
    extent=extent,
    norm=PowerNorm(gamma=1/3, vmin=vmin, vmax=vmax),
    aspect="auto",
)
ax.set_xlabel("Time [UTC]", fontsize=fontsize)
ax.set_ylabel("Elongation/LOS angle [degree]", fontsize=fontsize)
ax.set_title("PUNCH stack-plot | Left click: add | Right click: delete nearest", fontsize=fontsize)

locator = mdates.AutoDateLocator()
formatter = mdates.ConciseDateFormatter(locator)
ax.xaxis.set_major_locator(locator)
ax.xaxis.set_major_formatter(formatter)
fig.autofmt_xdate()

# Store clicked points and their plot artists
pts = []
markers = []   # parallel list — one Line2D per point

def redraw_markers():
    """Remove all marker artists and redraw from pts list."""
    for m in markers:
        m.remove()
    markers.clear()
    for (x, y) in pts:
        ln, = ax.plot(x, y, 'o', color='red', markersize=6, markeredgecolor='black')
        markers.append(ln)
    fig.canvas.draw()

def on_click(event):
    if not event.inaxes:
        return

    if event.button == 1:                          # --- Left click: ADD ---
        pts.append((event.xdata, event.ydata))
        ln, = ax.plot(event.xdata, event.ydata, 'o',
                      color='red', markersize=6, markeredgecolor='black')
        markers.append(ln)
        fig.canvas.draw()

    elif event.button == 3 and len(pts) > 0:       # --- Right click: DELETE nearest ---
        # Normalize axes scales so time and degree axes are comparable
        x_scale = extent[1] - extent[0]
        y_scale = extent[3] - extent[2]

        distances = [
            np.sqrt(((event.xdata - px) / x_scale) ** 2 +
                    ((event.ydata - py) / y_scale) ** 2)
            for (px, py) in pts
        ]
        idx = int(np.argmin(distances))
        pts.pop(idx)
        redraw_markers()          # clean redraw — no ghost markers

fig.canvas.mpl_connect('button_press_event', on_click)

done_btn = widgets.Button(description="Done selecting points")
output = widgets.Output()

def on_done(b):
    global x_coords_num, y_coords, x_coords_dt
    x_coords_num = np.array([p[0] for p in pts])
    y_coords     = np.array([p[1] for p in pts])
    x_coords_dt  = [mdates.num2date(x) for x in x_coords_num]
    with output:
        output.clear_output() # Echo the picked coordinates so they can be reused or copied into later analysis.
        print(f"Captured {len(pts)} points.")
        print("x datetime:", x_coords_dt)
        print("y degrees:", y_coords)

done_btn.on_click(on_done)
display(done_btn, output)


## Step 4: Convert the tracked feature into a speed estimate

The speed estimate here is derived directly from the user-selected track. It is useful for quick interpretation and classroom discussion, but it should not be treated as a publication-ready CME kinematic result without additional uncertainty analysis.


### Estimating Kinematics from the stack-plot

Once points are selected along a track in the stack-plot, we convert those clicks into approximate radial distances and times. A finite-difference estimate then gives an instantaneous speed profile.

This is a quick-look estimate intended for demonstration and exploration, not a full reconstruction of CME propagation geometry.


### ***Simple numerical differentiation to estimate instantaneous speed***

In [ ]:
t_array = np.array([(t - x_coords_dt[0]).total_seconds() for t in x_coords_dt])  # seconds since first click
r_array = np.array(np.sin(y_coords * u.deg)) * full_header["DSUN_OBS"] / 1000     # km

dt = np.diff(t_array)
dr = np.diff(r_array)
valid = dt > 0
v_instant = dr[valid] / dt[valid]  # km/s

t_mid = 0.5 * (t_array[:-1] + t_array[1:])


In [ ]:
# Plot the point-to-point speed estimate.
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(t_mid / 60.0, v_instant, 'o-')
ax.set_xlabel("Time [min]")
ax.set_ylabel("Speed [km/s]")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### ***Not So-simple numerical differentiation to estimate instantaneous speed***

In [ ]:
# Convert the selected stack-plot points into elapsed time and approximate radial distance.
# Here we use a simple trigonometric conversion based on elongation angle and observer distance.
# That is fine for a quick-look demo, but more careful geometry may be needed for science analysis.

t_array = np.array([(t - x_coords_dt[0]).total_seconds() for t in x_coords_dt])
r_array = np.array(np.sin(y_coords * u.deg)) * full_header["DSUN_OBS"] / 1000  # km

# --- Sort by time (safety, in case clicks were out of order) ---
order = np.argsort(t_array)
t_array = t_array[order]
r_array = r_array[order]

# ================================================================
# Method 1: Cubic Spline analytical derivative
# Fits a smooth spline through all points and evaluates dr/dt
# analytically — no finite differences at all.
# Best when your points are sparse but cleanly follow a smooth track.
# ================================================================
cs = CubicSpline(t_array, r_array)
t_dense = np.linspace(t_array[0], t_array[-1], 500)   # dense time grid
r_dense = cs(t_dense)                                  # interpolated r
v_spline = cs(t_dense, 1)                              # 1st derivative = dr/dt [km/s]

# ================================================================
# Method 2: Savitzky-Golay on the interpolated dense curve
# Fits a local polynomial over a sliding window — good at removing
# noise while preserving peak shapes (better than simple smoothing).
# Needs enough points: window_length must be odd and < len(t_dense).
# ================================================================
# Resample onto a uniform time grid first (SG requires uniform spacing)
t_uniform = np.linspace(t_array[0], t_array[-1], 500)
r_uniform = cs(t_uniform)          # use spline to get uniform r
dt_uniform = t_uniform[1] - t_uniform[0]

window = 251    # tune: larger = smoother, must be odd
polyorder = 5   # polynomial order (3 or 4 works well)
v_savgol = savgol_filter(r_uniform, window_length=window,
                         polyorder=polyorder, deriv=1, delta=dt_uniform)

# ================================================================
# Method 3: Gaussian-kernel weighted local linear regression
# (a type of Locally Weighted Scatterplot Smoothing; LOWESS)
# At each point, fits a line weighted by a Gaussian centred there.
# Most robust to outlier clicks; sigma controls the smoothing scale.
# ================================================================
def gaussian_local_derivative(t, r, sigma_seconds=None):
    """
    Estimate dr/dt at every point in t using Gaussian-weighted
    local linear regression.

    sigma_seconds : smoothing scale in seconds.
                    Defaults to ~20% of the total time span.
    """
    if sigma_seconds is None:
        sigma_seconds = 0.2 * (t[-1] - t[0])

    v = np.zeros_like(t, dtype=float)
    for i, ti in enumerate(t):
        w = np.exp(-0.5 * ((t - ti) / sigma_seconds) ** 2)
        # Weighted linear regression: r ≈ a + b*(t - ti)  →  b = dr/dt at ti
        W = np.diag(w)
        A = np.column_stack([np.ones_like(t), t - ti])
        try:
            coeffs = np.linalg.lstsq(W @ A, W @ r, rcond=None)[0]
            v[i] = coeffs[1]   # slope = instantaneous speed
        except np.linalg.LinAlgError:
            v[i] = np.nan
    return v

v_gaussian = gaussian_local_derivative(t_array, r_array)

# ================================================================
# Plot all three together for comparison
# ================================================================
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(t_dense / 60, v_spline,  '-',  color='steelblue', linewidth=2,
        label='Cubic spline derivative')
ax.plot(t_uniform / 60, v_savgol, '--', color='tomato',    linewidth=2,
        label=f'Savitzky-Golay (win={window}, order={polyorder})')
ax.plot(t_array / 60, v_gaussian, 'o-', color='seagreen',  linewidth=1.5,
        markersize=5, label='Gaussian local regression')

ax.set_xlabel("Time [min]", fontsize=fontsize)
ax.set_ylabel("Speed [km/s]", fontsize=fontsize)
ax.set_title("Instantaneous speed — differentiation methods compared", fontsize=fontsize)
ax.legend(fontsize=fontsize-4)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 5: Examine intensity evolution along a curved path

The final section uses a spline-defined slit to follow a feature through the stack-plot and inspect how its intensity changes along that track.


### Following a Curved Track Instead of Isolated Points

Some features are easier to characterize with a smooth path rather than point-by-point clicks. The next cells fit a spline through the selected path and extract the average intensity across a narrow curved slit.

That provides a simple way to examine how brightness evolves along a feature of interest.


In [ ]:
from scipy.interpolate import CubicSpline
from scipy.ndimage import map_coordinates

def extract_curved_slit(image, x_coords, y_coords, thickness=10, num_points=500):
    """
    Extract mean intensity along a curved slit defined by user-selected points.

    Parameters
    ----------
    image       : 2D numpy array
        Input stack-plot or any other 2D image.
    x_coords    : 1D array
        Clicked x positions in pixel units.
    y_coords    : 1D array
        Clicked y positions in pixel units.
    thickness   : int
        Half-width of the slit in pixels.
    num_points  : int
        Number of interpolated points used to sample the curve.

    Returns
    -------
    slit_1d     : 1D array
        Mean intensity sampled along the curved slit.
    curve_x     : 1D array
        Interpolated x coordinates of the curve.
    curve_y     : 1D array
        Interpolated y coordinates of the curve.
    """

    # Sort the clicked points in time before fitting the spline.
    order = np.argsort(x_coords)
    x_sorted = x_coords[order]
    y_sorted = y_coords[order]

    cs = CubicSpline(x_sorted, y_sorted)
    curve_x = np.linspace(x_sorted[0], x_sorted[-1], num_points)
    curve_y = cs(curve_x)

    # Tangent and normal vectors define the local slit direction.
    dx = np.gradient(curve_x)
    dy = np.gradient(curve_y)
    norm = np.sqrt(dx**2 + dy**2)
    tx = dx / norm
    ty = dy / norm
    nx = -ty
    ny = tx

    offsets = np.arange(-thickness, thickness + 1)
    slit_1d = np.zeros(num_points)

    for i in range(num_points):
        sample_x = curve_x[i] + offsets * nx[i]
        sample_y = curve_y[i] + offsets * ny[i]

        # map_coordinates expects coordinates in (row, column) order.
        intensities = map_coordinates(image, [sample_y, sample_x],
                                      order=1, mode='nearest')
        slit_1d[i] = np.nanmean(intensities)

    return slit_1d, curve_x, curve_y


In [ ]:
# Apply a simple r^3 radial weighting to boost visibility of faint outer features.
# This is to compensate the coronal inensity fall-off and identify if the CME is
# accumulating mass or getting diluted as it propagates.

#################################################################################
##############              Notice the change made here            ##############
#################################################################################
ny = distance_time_map.shape[0]
radial = np.linspace(0, 45, ny, endpoint=False)[:, None]
input_map = distance_time_map2 * radial**3
#################################################################################

vmin = np.percentile(input_map, 15)
vmax = np.percentile(input_map, 97)
fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(
    input_map,
    origin="lower",
    cmap=cmap_punch,
    extent=extent,
    norm=PowerNorm(gamma=1/2.2, vmin=vmin, vmax=vmax),
    aspect="auto",
)

ax.set_xlabel("Time [UTC]", fontsize=fontsize)
ax.set_ylabel("Elongation/LOS angle [degree]", fontsize=fontsize)
ax.set_title("PUNCH stack-plot", fontsize=fontsize)

locator = mdates.AutoDateLocator()
formatter = mdates.ConciseDateFormatter(locator)
ax.xaxis.set_major_locator(locator)
ax.xaxis.set_major_formatter(formatter)
fig.autofmt_xdate()

# Collect points that define the approximate path of the feature.
pts = []

def on_click(event):
    if event.inaxes and event.button == 1:
        pts.append((event.xdata, event.ydata))
        ax.plot(event.xdata, event.ydata, 'o', color='red',
                markersize=6, markeredgecolor='black')
        fig.canvas.draw()

fig.canvas.mpl_connect('button_press_event', on_click)

done_btn = widgets.Button(description="Done selecting points")
output = widgets.Output()

def on_done(b):
    global x_coords_num, y_coords, x_coords_dt, slit_1d, curve_x, curve_y

    x_coords_num = np.array([p[0] for p in pts])
    y_coords_raw = np.array([p[1] for p in pts])
    x_coords_dt = [mdates.num2date(x) for x in x_coords_num]

    # Convert plot coordinates back into pixel coordinates for image sampling.
    ncols = input_map.shape[1]
    t_min, t_max = t_num.min(), t_num.max()
    x_px = (x_coords_num - t_min) / (t_max - t_min) * (ncols - 1)

    nrows = input_map.shape[0]
    y_px = y_coords_raw / 45.0 * (nrows - 1)

    slit_1d, curve_x, curve_y = extract_curved_slit(
        input_map, x_px, y_px, thickness=5, num_points=500
    )

    with output:
        print(f"Captured {len(pts)} points.")
        print(f"Slit extracted: {slit_1d.shape[0]} points along curve")

done_btn.on_click(on_done)
display(done_btn, output)


In [ ]:
# Show the selected curved slit on top of the J-map and plot the extracted intensity profile.
fig2, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(input_map, origin="lower", cmap=cmap_punch,
               extent=extent, norm=PowerNorm(gamma=1/2.2, vmin=vmin, vmax=vmax),
               aspect="auto")

# Convert the spline path from pixel coordinates back into axis units for plotting.
t_min, t_max = t_num.min(), t_num.max()
ncols = input_map.shape[1]
nrows = input_map.shape[0]
curve_t = curve_x / (ncols - 1) * (t_max - t_min) + t_min
curve_deg = curve_y / (nrows - 1) * 45.0
axes[0].plot(curve_t, curve_deg, 'c--', linewidth=1.5, label='Slit path')
axes[0].xaxis.set_major_formatter(mdates.ConciseDateFormatter(mdates.AutoDateLocator()))
axes[0].set_title("Stack plot with slit path")
axes[0].set_ylabel("Elongation angle")
axes[0].legend()

axes[1].plot(slit_1d)
axes[1].set_xlabel("Point along slit")
axes[1].set_ylabel("Mean intensity")
axes[1].set_title("Intensity profile")

plt.tight_layout()
plt.show()
